# 🟢 **Generate Test**

In [ ]:
import os
import math
import torch
from torch import nn
from torch.nn import functional as F
from tokenizers import Tokenizer


BEST_MODEL_PATH = (
    "/data/logs/"
    "Alphabet_NLP_Dataset_20260906_063836/"
    "best_model.pt"
)

TOKENIZER_PATH = (
    "/data/tokenizer/"
    "bpe-tokenizer_alphabet_nlp_dataset.json"
)


TEST_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("=" * 80)
print("🧪 Independent Best Model Test")
print("=" * 80)

print(f"Device: {TEST_DEVICE}")
print(f"Tokenizer: {TOKENIZER_PATH}")
print(f"Model: {BEST_MODEL_PATH}")


if not os.path.exists(TOKENIZER_PATH):
    raise FileNotFoundError(
        f"❌ Tokenizer not found:\n{TOKENIZER_PATH}"
    )

if not os.path.exists(BEST_MODEL_PATH):
    raise FileNotFoundError(
        f"❌ Best model not found:\n{BEST_MODEL_PATH}"
    )

print("\n✅ Tokenizer file found.")
print("✅ Best model file found.")


test_tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

print("\n✅ Tokenizer loaded successfully.")

print(f"Vocabulary size: {test_tokenizer.get_vocab_size():,}")


class IndependentMultiHeadAttention(nn.Module):

    def __init__(self, n_embd, n_head, dropout_rate):
        super().__init__()

        self.n_embd = n_embd
        self.n_head = n_head
        self.head_size = n_embd // n_head

        self.qkv_proj = nn.Linear(
            n_embd,
            3 * n_embd,
            bias=False
        )

        self.c_proj = nn.Linear(
            n_embd,
            n_embd,
            bias=False
        )

        self.c_proj.residual = True

    def forward(self, x):

        B, T, C = x.shape

        q, k, v = (
            self.qkv_proj(x)
            .view(
                B,
                T,
                3 * self.n_head,
                self.head_size
            )
            .transpose(1, 2)
            .chunk(3, dim=-3)
        )

        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            is_causal=True
        )

        y = (
            y.transpose(1, 2)
            .contiguous()
            .view(B, T, C)
        )

        y = self.c_proj(y)

        return y


class IndependentFeedForward(nn.Module):

    def __init__(
        self,
        n_embd,
        f_expnd,
        dropout_rate
    ):
        super().__init__()

        hidden_size = int(
            f_expnd * n_embd
        )

        self.up_proj = nn.Linear(
            n_embd,
            hidden_size,
            bias=False
        )

        self.down_proj = nn.Linear(
            hidden_size,
            n_embd,
            bias=False
        )

        self.down_proj.residual = True

        self.mlp_dropout = nn.Dropout(
            dropout_rate
        )

    def forward(self, x):

        return self.mlp_dropout(
            self.down_proj(
                F.gelu(
                    self.up_proj(x)
                )
            )
        )


class IndependentDecoderBlock(nn.Module):

    def __init__(
        self,
        n_embd,
        n_head,
        f_expnd,
        dropout_rate
    ):
        super().__init__()

        self.ln1 = nn.LayerNorm(n_embd)

        self.mha = IndependentMultiHeadAttention(
            n_embd,
            n_head,
            dropout_rate
        )

        self.ln2 = nn.LayerNorm(n_embd)

        self.mlp = IndependentFeedForward(
            n_embd,
            f_expnd,
            dropout_rate
        )

        self.dropout = nn.Dropout(
            dropout_rate
        )

    def forward(self, x):

        x = x + self.dropout(
            self.mha(
                self.ln1(x)
            )
        )

        x = x + self.dropout(
            self.mlp(
                self.ln2(x)
            )
        )

        return x


class IndependentARK(nn.Module):

    def __init__(
        self,
        vocab_size=10000,
        max_seq_len=1024,
        n_layer=8,
        n_head=16,
        n_embd=128,
        f_expnd=4,
        dropout_rate=0.2
    ):
        super().__init__()

        self.wte = nn.Embedding(
            vocab_size,
            n_embd
        )

        self.wpe = nn.Embedding(
            max_seq_len,
            n_embd
        )

        self.decoders = nn.ModuleList(
            [
                IndependentDecoderBlock(
                    n_embd=n_embd,
                    n_head=n_head,
                    f_expnd=f_expnd,
                    dropout_rate=dropout_rate
                )
                for _ in range(n_layer)
            ]
        )

        self.lnf = nn.LayerNorm(n_embd)

        self.lm_head = nn.Linear(
            n_embd,
            vocab_size,
            bias=False
        )

        # Weight tying
        self.lm_head.weight = self.wte.weight

    def forward(self, idx):

        B, T = idx.shape

        positions = torch.arange(
            T,
            device=idx.device
        )

        x = (
            self.wte(idx)
            +
            self.wpe(positions)
        )

        for decoder in self.decoders:
            x = decoder(x)

        x = self.lnf(x)

        logits = self.lm_head(x)

        return logits


test_model = IndependentARK(
    vocab_size=10_000,
    max_seq_len=1024,
    n_layer=8,
    n_head=16,
    n_embd=128,
    f_expnd=4,
    dropout_rate=0.2
).to(TEST_DEVICE)


print("\n✅ Independent ARK architecture created.")

total_params = sum(
    p.numel()
    for p in test_model.parameters()
)

print(
    f"📊 Parameters: "
    f"{total_params:,} "
    f"({total_params / 1e6:.2f}M)"
)


print("\n🔄 Loading best_model.pt...")

checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=TEST_DEVICE
)


print("\n🔍 Checkpoint information")

if isinstance(checkpoint, dict):

    print(
        "Checkpoint keys:"
    )

    for key in checkpoint.keys():
        print(f"   • {key}")

else:

    raise TypeError(
        "❌ Unexpected checkpoint format."
    )


if "model_state_dict" not in checkpoint:

    raise KeyError(
        "❌ 'model_state_dict' not found in checkpoint."
    )

test_model.load_state_dict(
    checkpoint["model_state_dict"]
)

test_model.eval()

print(
    "\n✅ Best model loaded successfully."
)


checkpoint_state = checkpoint[
    "model_state_dict"
]

model_state = test_model.state_dict()

checkpoint_keys = set(
    checkpoint_state.keys()
)

model_keys = set(
    model_state.keys()
)

missing_keys = (
    model_keys -
    checkpoint_keys
)

unexpected_keys = (
    checkpoint_keys -
    model_keys
)

shape_mismatches = []

for key in checkpoint_keys & model_keys:

    if (
        checkpoint_state[key].shape
        !=
        model_state[key].shape
    ):

        shape_mismatches.append(
            (
                key,
                checkpoint_state[key].shape,
                model_state[key].shape
            )
        )


print("\n🔍 Checkpoint verification")

print(
    f"Checkpoint tensors: "
    f"{len(checkpoint_keys):,}"
)

print(
    f"Model tensors:      "
    f"{len(model_keys):,}"
)


if missing_keys:

    print("\n❌ Missing keys:")

    for key in missing_keys:
        print("   ", key)

else:

    print("✅ No missing keys.")


if unexpected_keys:

    print("\n⚠️ Unexpected keys:")

    for key in unexpected_keys:
        print("   ", key)

else:

    print("✅ No unexpected keys.")


if shape_mismatches:

    print("\n❌ Shape mismatches:")

    for (
        key,
        checkpoint_shape,
        model_shape
    ) in shape_mismatches:

        print(
            f"   {key}: "
            f"checkpoint={checkpoint_shape}, "
            f"model={model_shape}"
        )

else:

    print("✅ All tensor shapes match.")


if (
    not missing_keys
    and
    not unexpected_keys
    and
    not shape_mismatches
):

    print(
        "\n🎯 Checkpoint architecture "
        "matches the model perfectly."
    )

else:

    raise RuntimeError(
        "\n❌ Checkpoint is not fully compatible."
    )


def independent_generate(
    model,
    tokenizer,
    prompt,
    max_seq_len=128,
    temperature=0.7,
    top_k=10,
    greedy=False,
    device="cuda",
    seed=42
):

    model.eval()

    
    prompt_ids = tokenizer.encode(
        prompt
    ).ids

    if len(prompt_ids) == 0:

        raise ValueError(
            "❌ Prompt produced zero tokens."
        )

    if len(prompt_ids) >= max_seq_len:

        raise ValueError(
            "❌ Prompt is already equal to "
            "or longer than max_seq_len."
        )

    inputs = torch.tensor(
        prompt_ids,
        dtype=torch.long,
        device=device
    ).unsqueeze(0)


    rng = torch.Generator(
        device=device
    )

    rng.manual_seed(seed)


    with torch.no_grad():

        while inputs.shape[1] < max_seq_len:

            logits = model(inputs)

            next_logits = logits[
                :, -1, :
            ]


            if greedy:

                next_token = torch.argmax(
                    next_logits,
                    dim=-1
                )


            else:

                temperature = max(
                    temperature,
                    1e-5
                )

                probs = torch.softmax(
                    next_logits / temperature,
                    dim=-1
                )

                k = min(
                    top_k,
                    probs.shape[-1]
                )

                topk_probs, topk_indices = (
                    torch.topk(
                        probs,
                        k=k,
                        dim=-1
                    )
                )

                sampled = torch.multinomial(
                    topk_probs,
                    num_samples=1,
                    generator=rng
                )

                next_token = torch.gather(
                    topk_indices,
                    -1,
                    sampled
                ).squeeze(-1)


            inputs = torch.cat(
                [
                    inputs,
                    next_token.unsqueeze(-1)
                ],
                dim=-1
            )


    generated_ids = inputs[
        0,
        len(prompt_ids):
    ].tolist()

    generated_text = tokenizer.decode(
        generated_ids
    )

    return generated_text


TEST_PROMPT = (
    "بازاندیشی در هوش مصنوعی "
    "(Reflection in Artificial Intelligence) "
    "به مفهوم و فرایندی اشاره دارد که"
)


print("\n")
print("=" * 100)
print("🧪 TEST PROMPT")
print("=" * 100)

print(TEST_PROMPT)


print("\n")
print("=" * 100)
print("🧠 GREEDY DECODING")
print("=" * 100)

greedy_output = independent_generate(
    model=test_model,
    tokenizer=test_tokenizer,
    prompt=TEST_PROMPT,
    max_seq_len=128,
    greedy=True,
    device=TEST_DEVICE
)

print("\n[Prompt]")
print(TEST_PROMPT)

print("\n[Generated continuation]")
print(greedy_output)


print("\n")
print("=" * 100)
print("🎲 TOP-K SAMPLING")
print("=" * 100)

for sample_number in range(1, 4):

    output = independent_generate(
        model=test_model,
        tokenizer=test_tokenizer,
        prompt=TEST_PROMPT,
        max_seq_len=128,
        temperature=0.7,
        top_k=10,
        greedy=False,
        device=TEST_DEVICE,
        seed=42 + sample_number
    )

    print(
        f"\n[Sample {sample_number}]"
    )

    print(
        output
    )

print("\n")
print("=" * 100)
print("✅ TEST COMPLETED")
print("=" * 100)